# **02 - Construcción, Diagnóstico y Refinamiento de Modelos**
## **Proyecto: Impacto de la Inteligencia Artificial Generativa en Estudiantes**

Este notebook aborda las etapas centrales de modelamiento y optimización:
1. **División de datos y validación estadística del Split**.
2. **Entrenamiento de modelos**: Baseline, Lineal Múltiple y Polinomial.
3. **Diagnóstico con Validación Cruzada** (`cross_val_score` y `cross_val_predict`).
4. **Regularización**: Ridge y Lasso.
5. **Optimización**: Búsqueda en grilla (`GridSearchCV`).
6. **Serialización**: Almacenar los modelos entrenados en `models/`.

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL E IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import os
import sys
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# Agregar directorio raíz para importar el paquete src
sys.path.append(os.path.abspath('..'))

import src.preparation as prep
import src.plots as plots

print('Entorno de trabajo inicializado correctamente.')

### 1. Ingesta y División Train/Test

Cargamos los datos procesados y creamos conjuntos de entrenamiento y prueba reproducibles.

In [ ]:
df = pd.read_csv('../data/processed/ai_student_impact_procesado.csv')

X = df.drop(columns=['Post_Semester_GPA'])
y = df['Post_Semester_GPA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'Entrenamiento: {X_train.shape[0]} registros | Prueba: {X_test.shape[0]} registros')

### 2. Validación Estadística de Representatividad del Split

Auditemos el split usando nuestra función modularizada en `src/preparation.py`.

In [ ]:
prep.validate_split(y_train, y_test)

### 3. Modelo Baseline (Regresión Simple)

Utiliza únicamente `Pre_Semester_GPA` para la predicción.

In [ ]:
X_train_base = X_train[['Pre_Semester_GPA']]
X_test_base = X_test[['Pre_Semester_GPA']]

lm_base = LinearRegression()
lm_base.fit(X_train_base, y_train)

y_pred_train_b = lm_base.predict(X_train_base)
y_pred_test_b = lm_base.predict(X_test_base)

print('=== MODELO BASELINE ===')
print(f'Train R² : {r2_score(y_train, y_pred_train_b):.4f} | Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train_b)):.4f}')
print(f'Test  R² : {r2_score(y_test, y_pred_test_b):.4f} | Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test_b)):.4f}')

### 4. Regresión Lineal Múltiple

Ajusta una regresión sobre todas las variables predictoras procesadas.

In [ ]:
variables_numericas = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Traditional_Study_Hours', 
                        'Anxiety_Level_During_Exams', 'Tool_Diversity', 'Perceived_AI_Dependency']
variables_categoricas = ['Major_Category', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Paid_Subscription']

# Creamos el preprocesador modularizado sin expansión polinomial
preprocessor = prep.create_preprocessor(variables_numericas, variables_categoricas)

pipeline_multi = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

pipeline_multi.fit(X_train, y_train)

y_pred_train_m = pipeline_multi.predict(X_train)
y_pred_test_m = pipeline_multi.predict(X_test)

print('=== REGRESIÓN LINEAL MÚLTIPLE ===')
print(f'Train R² : {r2_score(y_train, y_pred_train_m):.4f} | Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train_m)):.4f}')
print(f'Test  R² : {r2_score(y_test, y_pred_test_m):.4f} | Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test_m)):.4f}')

### 5. Regresión Polinomial (Grado 2)

Ajusta interacciones de segundo grado para las variables continuas.

In [ ]:
# Preprocesador modular con grado 2 para numéricas
preprocessor_poly = prep.create_preprocessor(variables_numericas, variables_categoricas, poly_degree=2)

pipeline_poly = Pipeline(steps=[
    ('preprocessor', preprocessor_poly),
    ('regressor', LinearRegression())
])

pipeline_poly.fit(X_train, y_train)

y_pred_train_p = pipeline_poly.predict(X_train)
y_pred_test_p = pipeline_poly.predict(X_test)

print('=== REGRESIÓN POLINOMIAL GRADO 2 ===')
print(f'Train R² : {r2_score(y_train, y_pred_train_p):.4f} | Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train_p)):.4f}')
print(f'Test  R² : {r2_score(y_test, y_pred_test_p):.4f} | Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test_p)):.4f}')

# Graficamos la distribución de los residuos usando la función en src/plots
plots.plot_residuals_distribution(y_test, y_pred_test_m, y_pred_test_p, filepath='../outputs/figures/residuals_distribution.png')

### 6. Diagnóstico del Modelo: Validación Cruzada vs Hold-Out

Evaluaremos un modelo polinomial de grado 3 de alta complejidad para auditar si sufre de sobreajuste catastrófico.

In [ ]:
preprocessor_complex = prep.create_preprocessor(variables_numericas, variables_categoricas, poly_degree=3)
model_complex = Pipeline(steps=[
    ('preprocessor', preprocessor_complex),
    ('regressor', LinearRegression())
])

# Hold-Out
model_complex.fit(X_train, y_train)
pred_test = model_complex.predict(X_test)
print(f'R² Hold-Out (Prueba): {r2_score(y_test, pred_test):.4f}')

# K-Fold Cross Validation
cv_scores = cross_val_score(model_complex, X, y, cv=5, scoring='r2')
print('\nEvaluación K-Fold Cross Validation:')
for idx, score in enumerate(cv_scores, start=1):
    print(f'  Fold {idx}: R² = {score:.4f}')
print(f'Promedio R² CV: {cv_scores.mean():.4f} | Desviación Estándar: {cv_scores.std():.4f}')

#### Predicciones Out-of-Fold con `cross_val_predict()`

In [ ]:
# Definimos un pipeline equivalente para el Baseline para poder usarlo en cross-validation
pipeline_baseline = Pipeline(steps=[
    ('scaler', prep.create_preprocessor(['Pre_Semester_GPA'], [])),
    ('regressor', LinearRegression())
])

y_cv_pred_base = cross_val_predict(pipeline_baseline, X_train[['Pre_Semester_GPA']], y_train, cv=5)
y_cv_pred_multi = cross_val_predict(pipeline_multi, X_train, y_train, cv=5)
y_cv_pred_poly = cross_val_predict(pipeline_poly, X_train, y_train, cv=5)

r2_cv_base = r2_score(y_train, y_cv_pred_base)
rmse_cv_base = np.sqrt(mean_squared_error(y_train, y_cv_pred_base))
r2_cv_multi = r2_score(y_train, y_cv_pred_multi)
rmse_cv_multi = np.sqrt(mean_squared_error(y_train, y_cv_pred_multi))
r2_cv_poly = r2_score(y_train, y_cv_pred_poly)
rmse_cv_poly = np.sqrt(mean_squared_error(y_train, y_cv_pred_poly))

plots.plot_cv_predictions(y_train, y_cv_pred_base, y_cv_pred_multi, y_cv_pred_poly,
                          r2_cv_base, rmse_cv_base, r2_cv_multi, rmse_cv_multi, r2_cv_poly, rmse_cv_poly,
                          filepath='../outputs/figures/cv_predictions_diagnostic.png')

### 7. Control de Complejidad: Regularización (Ridge y Lasso)

Evaluaremos Ridge ($lpha=10.0$) y Lasso ($lpha=0.01$) para mitigar el sobreajuste polinomial.

In [ ]:
pipeline_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor_poly),
    ('regressor', Ridge(alpha=10.0))
])

pipeline_lasso = Pipeline(steps=[
    ('preprocessor', preprocessor_poly),
    ('regressor', Lasso(alpha=0.01))
])

pipeline_ridge.fit(X_train, y_train)
pipeline_lasso.fit(X_train, y_train)

print('=== MODELO RIDGE (L2) ===')
print(f'Test R² : {r2_score(y_test, pipeline_ridge.predict(X_test)):.4f}')

print('\n=== MODELO LASSO (L1) ===')
print(f'Test R² : {r2_score(y_test, pipeline_lasso.predict(X_test)):.4f}')

### 8. Optimización Automática usando GridSearchCV

In [ ]:
# Preprocesador dinámico para GridSearchCV
preprocessor_grid = prep.create_preprocessor(variables_numericas, variables_categoricas, poly_degree=1)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_grid),
    ('ridge', Ridge())
])

param_grid = {
    'preprocessor__num__poly__degree': [1, 2],
    'ridge__alpha': [0.1, 1.0, 10.0, 100.0]
}

grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Búsqueda en grilla completada.')
print(f'Mejores Parámetros: {grid_search.best_params_}')
print(f'Mejor R² en CV: {grid_search.best_score_:.4f}')

### 9. Serialización y Almacenamiento de Modelos

Guardamos todos los modelos ajustados en la carpeta `models/` como archivos `.pkl`.

In [ ]:
best_model = grid_search.best_estimator_

# Guardar modelos
joblib.dump(lm_base, '../models/baseline_model.pkl')
joblib.dump(pipeline_multi, '../models/multiple_linear_model.pkl')
joblib.dump(pipeline_poly, '../models/polynomial_linear_model.pkl')
joblib.dump(pipeline_ridge, '../models/ridge_model.pkl')
joblib.dump(pipeline_lasso, '../models/lasso_model.pkl')
joblib.dump(best_model, '../models/optimized_best_model.pkl')

print('Todos los modelos se han guardado exitosamente en ../models/')